# Goal Behaviour Resolution: Correct Response Model

Consequent Reversibility was not included in the final escape models because, in this subgroup structure, it is partially encoded by Goal_Type and produced unstable estimates when entered together with Goal_Type.

Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Overall Subgroup
This subset of data explores observations when _Focus_ = I and _Focus_= They.
1. Create resolved_correct
2. Counts and Proportions for Goals vs. no_goals
3. Logistic Models
    - RC1: Agency (goal_frequent vs. goal_non_frequent)
    - RC2: Goal Type (goal_frequent vs. goal_non_frequent)
    - RC3: Additive Model (goal_frequent vs. goal_non_frequent)
    - RC4: Interactive Model (goal_frequent vs. goal_non_frequent)
    - RC5: Agency  (no_goals)
4. Model Comparison
5. Summary Table

Read Data

In [2]:
subgroup_theoretical = pd.read_csv("../../data/processed/subgroup_theoretical.csv")

1. Create resolved_correct

Binary Outcome: include correct responses from escape_L2 observations.

In [3]:
subgroup_theoretical["escape_L2"] = (
    subgroup_theoretical["Response_Full"] != "L2_other"
).astype(int)

In [4]:
escapees = subgroup_theoretical[subgroup_theoretical["escape_L2"] == 1].copy()

In [5]:
escapees["resolved_correct"] = (escapees["Response_Full"] == "correct").astype(int)

In [6]:
goals_escapees = escapees[escapees["Goal_Type"] != "no_goal"].copy()

In [7]:
no_goal_escapees = escapees[escapees["Goal_Type"] == "no_goal"].copy()

2. Counts and Proportions

In [8]:
resolved_correct_counts = pd.crosstab(
    escapees["resolved_correct"],
    escapees["Response_Full"])

resolved_correct_counts

Response_Full,L1_transfer,correct,missing_response
resolved_correct,,,
0,96,0,3
1,0,61,0


In [9]:
goals_resolved_correct_counts = pd.crosstab(
    goals_escapees["resolved_correct"],
    goals_escapees["Response_Full"])

goals_resolved_correct_counts

Response_Full,L1_transfer,correct,missing_response
resolved_correct,,,
0,62,0,1
1,0,43,0


In [10]:
no_goal_resolved_correct_counts = pd.crosstab(
    no_goal_escapees["resolved_correct"],
    no_goal_escapees["Response_Full"])

no_goal_resolved_correct_counts

Response_Full,L1_transfer,correct,missing_response
resolved_correct,,,
0,34,0,2
1,0,18,0


In [11]:
resolved_correct_props = escapees["resolved_correct"].value_counts(normalize = True)
resolved_correct_props

resolved_correct
0    0.61875
1    0.38125
Name: proportion, dtype: float64

From 160 observations, almost 38% of responses that had escaped *L2_other* options were resolved correctly. 

Counts and Proportions by Condition:

In [12]:
counts_condition = pd.crosstab(
    [escapees["Goal_Type"], escapees["Agent"]],
    escapees["resolved_correct"],
    margins = True)

counts_condition

resolved_correct          0   1  All
Goal_Type         Agent             
goal_frequent     0      14  14   28
                  1      12  10   22
goal_non_frequent 0      21  12   33
                  1      16   7   23
no_goal           0      25   9   34
                  1      11   9   20
All                      99  61  160

In [13]:
goals_counts_condition = pd.crosstab(
    [goals_escapees["Goal_Type"], goals_escapees["Agent"]],
    goals_escapees["resolved_correct"],
    margins = True)

goals_counts_condition

resolved_correct          0   1  All
Goal_Type         Agent             
goal_frequent     0      14  14   28
                  1      12  10   22
goal_non_frequent 0      21  12   33
                  1      16   7   23
All                      63  43  106

In [14]:
no_goal_counts_condition = pd.crosstab(
    [no_goal_escapees["Goal_Type"], no_goal_escapees["Agent"]],
    no_goal_escapees["resolved_correct"],
    margins = True)

no_goal_counts_condition

resolved_correct   0   1  All
Goal_Type Agent              
no_goal   0       25   9   34
          1       11   9   20
All               36  18   54

In [15]:
props_condition = pd.crosstab(
    [escapees["Goal_Type"], escapees["Agent"]],
    escapees["resolved_correct"],
    normalize= "index")

props_condition

resolved_correct                0         1
Goal_Type         Agent                    
goal_frequent     0      0.500000  0.500000
                  1      0.545455  0.454545
goal_non_frequent 0      0.636364  0.363636
                  1      0.695652  0.304348
no_goal           0      0.735294  0.264706
                  1      0.550000  0.450000

In [16]:
goals_props_condition = pd.crosstab(
    [goals_escapees["Goal_Type"], goals_escapees["Agent"]],
    goals_escapees["resolved_correct"],
    normalize= "index")

goals_props_condition

resolved_correct                0         1
Goal_Type         Agent                    
goal_frequent     0      0.500000  0.500000
                  1      0.545455  0.454545
goal_non_frequent 0      0.636364  0.363636
                  1      0.695652  0.304348

In [17]:
no_goal_props_condition = pd.crosstab(
    [no_goal_escapees["Goal_Type"], no_goal_escapees["Agent"]],
    no_goal_escapees["resolved_correct"],
    normalize= "index")

no_goal_props_condition

resolved_correct         0         1
Goal_Type Agent                     
no_goal   0       0.735294  0.264706
          1       0.550000  0.450000

COMMENT GENERAL PROPORTIONS

3. Logistic Models

Import Libraries

In [18]:
import statsmodels.formula.api as smf
from scipy.stats import chi2

In [19]:
escapees["resolved_correct"].value_counts()

resolved_correct
0    99
1    61
Name: count, dtype: int64

In [20]:
goals_escapees["resolved_correct"].value_counts()

resolved_correct
0    63
1    43
Name: count, dtype: int64

In [21]:
no_goal_escapees["resolved_correct"].value_counts()

resolved_correct
0    36
1    18
Name: count, dtype: int64

**Model RC1**

resolved_correct ~ Agent

In [22]:
RC1 = smf.logit(
    "resolved_correct ~ Agent",
    data=goals_escapees
    ).fit()

print(RC1.summary())

Optimization terminated successfully.
         Current function value: 0.674048
         Iterations 4
                           Logit Regression Results                           
Dep. Variable:       resolved_correct   No. Observations:                  106
Model:                          Logit   Df Residuals:                      104
Method:                           MLE   Df Model:                            1
Date:                Tue, 07 Jul 2026   Pseudo R-squ.:                0.001766
Time:                        12:19:52   Log-Likelihood:                -71.449
converged:                       True   LL-Null:                       -71.575
Covariance Type:            nonrobust   LLR p-value:                    0.6151
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.2973      0.259     -1.148      0.251      -0.805       0.210
Agent         -0.2017      0.

**Model RC2**

resolved_correct ~ Goal_Type

In [23]:
RC2 = smf.logit(
    "resolved_correct ~ Goal_Type",
    data= goals_escapees
    ).fit()

print(RC2.summary())

Optimization terminated successfully.
         Current function value: 0.664988
         Iterations 4
                           Logit Regression Results                           
Dep. Variable:       resolved_correct   No. Observations:                  106
Model:                          Logit   Df Residuals:                      104
Method:                           MLE   Df Model:                            1
Date:                Tue, 07 Jul 2026   Pseudo R-squ.:                 0.01518
Time:                        12:19:52   Log-Likelihood:                -70.489
converged:                       True   LL-Null:                       -71.575
Covariance Type:            nonrobust   LLR p-value:                    0.1404
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept                         -0.0800      0.283     -0.283     

**Model RC3**

resolved_correct ~ Agent + Goal_Type

In [24]:
RC3 = smf.logit(
    "resolved_correct ~ Agent + Goal_Type",
    data= goals_escapees
    ).fit()

print(RC3.summary())

Optimization terminated successfully.
         Current function value: 0.663549
         Iterations 4
                           Logit Regression Results                           
Dep. Variable:       resolved_correct   No. Observations:                  106
Model:                          Logit   Df Residuals:                      103
Method:                           MLE   Df Model:                            2
Date:                Tue, 07 Jul 2026   Pseudo R-squ.:                 0.01731
Time:                        12:19:52   Log-Likelihood:                -70.336
converged:                       True   LL-Null:                       -71.575
Covariance Type:            nonrobust   LLR p-value:                    0.2896
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept                          0.0183      0.335      0.055     

**Model RC4**

resolved_correct ~ Agent * Goal_Type

In [25]:
RC4 = smf.logit(
    "resolved_correct ~ Agent * Goal_Type",
    data=goals_escapees
    ).fit()

print(RC4.summary())

Optimization terminated successfully.
         Current function value: 0.663498
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:       resolved_correct   No. Observations:                  106
Model:                          Logit   Df Residuals:                      102
Method:                           MLE   Df Model:                            3
Date:                Tue, 07 Jul 2026   Pseudo R-squ.:                 0.01739
Time:                        12:19:52   Log-Likelihood:                -70.331
converged:                       True   LL-Null:                       -71.575
Covariance Type:            nonrobust   LLR p-value:                    0.4772
                                           coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------
Intercept                            -1.226e-16      0.3

**Model RC5 (no_goals only)**

resolved_correct ~ Agent 

In [26]:
RC5 = smf.logit(
    "resolved_correct ~ Agent",
    data=no_goal_escapees
    ).fit()

print(RC5.summary())

Optimization terminated successfully.
         Current function value: 0.618743
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:       resolved_correct   No. Observations:                   54
Model:                          Logit   Df Residuals:                       52
Method:                           MLE   Df Model:                            1
Date:                Tue, 07 Jul 2026   Pseudo R-squ.:                 0.02792
Time:                        12:19:52   Log-Likelihood:                -33.412
converged:                       True   LL-Null:                       -34.372
Covariance Type:            nonrobust   LLR p-value:                    0.1659
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -1.0217      0.389     -2.628      0.009      -1.784      -0.260
Agent          0.8210      0.

4. Model Comparisons

In [27]:
from scipy.stats import chi2

def lr_test(model_small, model_large):

    LR = 2 * (model_large.llf - model_small.llf)

    df = model_large.df_model - model_small.df_model

    p = chi2.sf(LR, df)

    return pd.Series({
        "LR": LR,
        "df": df,
        "p_value": p
    })

In [28]:
lr_test(RC1, RC3)

LR         2.225701
df         1.000000
p_value    0.135731
dtype: float64

In [29]:
lr_test(RC1, RC4)

LR         2.236547
df         2.000000
p_value    0.326844
dtype: float64

5. Summary Table

In [30]:
goal_behaviour = ["Yes", "Yes", "Yes", "Yes", "Yes", "Yes", "Yes", "Yes"]

model_name = ["RC1", "RC2", "RC3", "RC3", "RC4", "RC4", "RC4", "RC5"]

outcome = ["resolved_correct","resolved_correct", "resolved_correct", "resolved_correct", "resolved_correct", "resolved_correct", "resolved_correct", "resolved_correct"]

subgroup = ["Overall", "Overall", "Overall", "Overall", "Overall", "Overall", "Overall", "Overall"]

formula = ["resolved_correct ~ Agent", "resolved_correct ~ Goal_Type", "resolved_correct ~ Agent + Goal_Type", "resolved_correct ~ Agent + Goal_Type",  "resolved_correct ~ Agent * Goal_Type", "resolved_correct ~ Agent * Goal_Type","resolved_correct ~ Agent * Goal_Type" , "resolved_correct ~ Agent"]

predictor= ["Agent", "goal_non_frequent", "goal_non_frequent", "Agent",  "goal_non_frequent", "Agent", "Agent:goal_non_frequent", "Agent"]

coef = [-0.2017, -0.5864, -0.5947, -0.2241, -0.5596, -0.1823, -0.0847, 0.8210]

std_error = [0.402, 0.400, 0.401, 0.407, 0.523, 0.571, 0.814, 0.594]

coef_p = [0.616, 0.142, 0.138, 0.582, 0.285, 0.750, 0.917, 0.167]

LLR_p = [0.6151, 0.1404, 0.2896, 0.2896, 0.4772, 0.4772, 0.4772, 0.1659]

observations = [106, 106, 106, 106, 106, 106, 106, 54]

Takeaway = []

In [31]:
gb_correct_summary_overall = pd.DataFrame({
    "Goal_Behaviour": goal_behaviour,
    "Model": model_name,
    "Outcome": outcome,
    "Subgroup": subgroup,
    "Formula": formula, 
    "Predictor": predictor,
    "Coef": coef,
    "Std_err": std_error,
    "Coef_p": coef_p,
    "Model_LLR_p": LLR_p,
    "N": observations,
})

gb_correct_summary_overall

,Goal_Behaviour,Model,Outcome,Subgroup,Formula,Predictor,Coef,Std_err,Coef_p,Model_LLR_p,N
0,Yes,RC1,resolved_correct,Overall,resolved_correct ~ Agent,Agent,-0.2017,0.402,0.616,0.6151,106
1,Yes,RC2,resolved_correct,Overall,resolved_correct ~ Goal_Type,goal_non_frequent,-0.5864,0.400,0.142,0.1404,106
2,Yes,RC3,resolved_correct,Overall,resolved_correct ~ Agent + Goal_Type,goal_non_frequent,-0.5947,0.401,0.138,0.2896,106
3,Yes,RC3,resolved_correct,Overall,resolved_correct ~ Agent + Goal_Type,Agent,-0.2241,0.407,0.582,0.2896,106
4,Yes,RC4,resolved_correct,Overall,resolved_correct ~ Agent * Goal_Type,goal_non_frequent,-0.5596,0.523,0.285,0.4772,106
5,Yes,RC4,resolved_correct,Overall,resolved_correct ~ Agent * Goal_Type,Agent,-0.1823,0.571,0.750,0.4772,106
6,Yes,RC4,resolved_correct,Overall,resolved_correct ~ Agent * Goal_Type,Agent:goal_non_frequent,-0.0847,0.814,0.917,0.4772,106
7,Yes,RC5,resolved_correct,Overall,resolved_correct ~ Agent,Agent,0.8210,0.594,0.167,0.1659,54


## Actors Subgroup
This subset of data explores observations only when _Focus_ = I.
1. Create resolved_correct for goals and no_goals
2. Counts and Proportions for goals and no_goals
3. Logistic Models
    - RC1_A: Agency (goal_frequent vs. goal_non_frequent)
    - RC2_A: Goal Type (goal_frequent vs. goal_non_frequent)
    - RC3_A: Additive Model (goal_frequent vs. goal_non_frequent)
    - RC4_A: Interactive Model (goal_frequent vs. goal_non_frequent)
    - RC5_A: Agency (no_goal)
4. Model Comparison
5. Summary Table

Read Data

In [32]:
subgroup_actors = subgroup_theoretical[subgroup_theoretical["Focus"] == "I"].copy()

1. Create resolved_correct

Binary Outcome: include correct responses from escape_L2 observations within actors.

In [33]:
subgroup_actors["escape_L2"] = (
    subgroup_actors["Response_Full"] != "L2_other"
).astype(int)

In [34]:
escapees_actors = subgroup_actors[subgroup_actors["escape_L2"] == 1].copy()

In [35]:
escapees_actors["resolved_correct"] = (escapees_actors["Response_Full"] == "correct").astype(int)

In [36]:
goals_escapees_actors = escapees_actors[escapees_actors["Goal_Type"] != "no_goal"].copy()

In [37]:
no_goal_escapees_actors = escapees_actors[escapees_actors["Goal_Type"] == "no_goal"].copy()

2. Counts and Proportions

In [38]:
resolved_correct_counts = pd.crosstab(
    escapees_actors["resolved_correct"],
    escapees_actors["Response_Full"])

resolved_correct_counts

Response_Full,L1_transfer,correct,missing_response
resolved_correct,,,
0,53,0,3
1,0,40,0


In [39]:
goals_resolved_correct_counts = pd.crosstab(
    goals_escapees_actors["resolved_correct"],
    goals_escapees_actors["Response_Full"])

goals_resolved_correct_counts

Response_Full,L1_transfer,correct,missing_response
resolved_correct,,,
0,31,0,1
1,0,29,0


In [40]:
no_goal_resolved_correct_counts = pd.crosstab(
    no_goal_escapees_actors["resolved_correct"],
    no_goal_escapees_actors["Response_Full"])

no_goal_resolved_correct_counts

Response_Full,L1_transfer,correct,missing_response
resolved_correct,,,
0,22,0,2
1,0,11,0


In [41]:
resolved_correct_props = escapees_actors["resolved_correct"].value_counts(normalize = True)
resolved_correct_props

resolved_correct
0    0.583333
1    0.416667
Name: proportion, dtype: float64

From 96 actors' observations (from the original 160), almost 41% of responses that managed to escape *L2_other* options were resolved correctly.

In [42]:
goals_resolved_correct_props = goals_escapees_actors["resolved_correct"].value_counts(normalize = True)
goals_resolved_correct_props

resolved_correct
0    0.52459
1    0.47541
Name: proportion, dtype: float64

In [43]:
no_goal_resolved_correct_props = no_goal_escapees_actors["resolved_correct"].value_counts(normalize = True)
no_goal_resolved_correct_props

resolved_correct
0    0.685714
1    0.314286
Name: proportion, dtype: float64

Counts and Proportions by Condition:

In [44]:
counts_condition = pd.crosstab(
    [escapees_actors["Goal_Type"], escapees_actors["Agent"]],
    escapees_actors["resolved_correct"],
    margins = True)

counts_condition

resolved_correct          0   1  All
Goal_Type         Agent             
goal_frequent     0       7   9   16
                  1       7   7   14
goal_non_frequent 0       7   9   16
                  1      11   4   15
no_goal           0      15   6   21
                  1       9   5   14
All                      56  40   96

In [45]:
goals_counts_condition = pd.crosstab(
    [goals_escapees_actors["Goal_Type"], goals_escapees_actors["Agent"]],
    goals_escapees_actors["resolved_correct"],
    margins = True)

goals_counts_condition

resolved_correct          0   1  All
Goal_Type         Agent             
goal_frequent     0       7   9   16
                  1       7   7   14
goal_non_frequent 0       7   9   16
                  1      11   4   15
All                      32  29   61

In [46]:
no_goal_counts_condition = pd.crosstab(
    no_goal_escapees_actors["Agent"],
    no_goal_escapees_actors["resolved_correct"],
    margins = True)

no_goal_counts_condition

resolved_correct,0,1,All
Agent,,,
0,15,6,21
1,9,5,14
All,24,11,35


In [47]:
props_condition = pd.crosstab(
    [escapees_actors["Goal_Type"], escapees_actors["Agent"]],
    escapees_actors["resolved_correct"],
    normalize= "index")

props_condition

resolved_correct                0         1
Goal_Type         Agent                    
goal_frequent     0      0.437500  0.562500
                  1      0.500000  0.500000
goal_non_frequent 0      0.437500  0.562500
                  1      0.733333  0.266667
no_goal           0      0.714286  0.285714
                  1      0.642857  0.357143

ADD COMMENT ON GENERAL PROPORTIONS 

In [48]:
goals_props_condition = pd.crosstab(
    [goals_escapees_actors["Goal_Type"], goals_escapees_actors["Agent"]],
    goals_escapees_actors["resolved_correct"],
    normalize= "index")

goals_props_condition

resolved_correct                0         1
Goal_Type         Agent                    
goal_frequent     0      0.437500  0.562500
                  1      0.500000  0.500000
goal_non_frequent 0      0.437500  0.562500
                  1      0.733333  0.266667

In [49]:
no_goal_props_condition = pd.crosstab(
    no_goal_escapees_actors["Agent"],
    no_goal_escapees_actors["resolved_correct"],
    normalize= "index")

no_goal_props_condition

resolved_correct,0,1
Agent,,
0,0.714286,0.285714
1,0.642857,0.357143


3. Logistic Models

In [50]:
escapees_actors["resolved_correct"].value_counts()

resolved_correct
0    56
1    40
Name: count, dtype: int64

In [51]:
goals_escapees_actors["resolved_correct"].value_counts()

resolved_correct
0    32
1    29
Name: count, dtype: int64

In [52]:
no_goal_escapees_actors["resolved_correct"].value_counts()

resolved_correct
0    24
1    11
Name: count, dtype: int64

**RC1_A Model** 

resolved_correct ~ Agent

In [53]:
RC1_A = smf.logit(
    "resolved_correct ~ Agent",
    data=goals_escapees_actors
    ).fit()

print(RC1_A.summary())

Optimization terminated successfully.
         Current function value: 0.675051
         Iterations 4
                           Logit Regression Results                           
Dep. Variable:       resolved_correct   No. Observations:                   61
Model:                          Logit   Df Residuals:                       59
Method:                           MLE   Df Model:                            1
Date:                Tue, 07 Jul 2026   Pseudo R-squ.:                 0.02440
Time:                        12:19:52   Log-Likelihood:                -41.178
converged:                       True   LL-Null:                       -42.208
Covariance Type:            nonrobust   LLR p-value:                    0.1512
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.2513      0.356      0.705      0.481      -0.447       0.950
Agent         -0.7438      0.

**RC2_A Model** 

resolved_correct ~  Goal_Type

In [54]:
RC2_A = smf.logit(
    "resolved_correct ~ Goal_Type",
    data=goals_escapees_actors
    ).fit()

print(RC2_A.summary())

Optimization terminated successfully.
         Current function value: 0.685414
         Iterations 4
                           Logit Regression Results                           
Dep. Variable:       resolved_correct   No. Observations:                   61
Model:                          Logit   Df Residuals:                       59
Method:                           MLE   Df Model:                            1
Date:                Tue, 07 Jul 2026   Pseudo R-squ.:                0.009427
Time:                        12:19:52   Log-Likelihood:                -41.810
converged:                       True   LL-Null:                       -42.208
Covariance Type:            nonrobust   LLR p-value:                    0.3723
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept                          0.1335      0.366      0.365     

**RC3_A Model**

resolved_correct ~  Agent + Goal_Type

In [55]:
RC3_A = smf.logit(
    "resolved_correct ~ Agent + Goal_Type",
    data=goals_escapees_actors
    ).fit()

print(RC3_A.summary())

Optimization terminated successfully.
         Current function value: 0.668669
         Iterations 4
                           Logit Regression Results                           
Dep. Variable:       resolved_correct   No. Observations:                   61
Model:                          Logit   Df Residuals:                       58
Method:                           MLE   Df Model:                            2
Date:                Tue, 07 Jul 2026   Pseudo R-squ.:                 0.03363
Time:                        12:19:52   Log-Likelihood:                -40.789
converged:                       True   LL-Null:                       -42.208
Covariance Type:            nonrobust   LLR p-value:                    0.2419
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept                          0.4856      0.449      1.081     

**RC4_A Model**

resolved_correct ~ Agent * Goal_Type

In [56]:
RC4_A = smf.logit(
    "resolved_correct ~ Agent * Goal_Type",
    data= goals_escapees_actors
    ).fit()

print(RC4_A.summary())

Optimization terminated successfully.
         Current function value: 0.661194
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:       resolved_correct   No. Observations:                   61
Model:                          Logit   Df Residuals:                       57
Method:                           MLE   Df Model:                            3
Date:                Tue, 07 Jul 2026   Pseudo R-squ.:                 0.04443
Time:                        12:19:52   Log-Likelihood:                -40.333
converged:                       True   LL-Null:                       -42.208
Covariance Type:            nonrobust   LLR p-value:                    0.2897
                                           coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------
Intercept                                0.2513      0.5

In [57]:
RC5_A = smf.logit(
    "resolved_correct ~ Agent",
    data= no_goal_escapees_actors
    ).fit()

print(RC5_A.summary())

Optimization terminated successfully.
         Current function value: 0.619664
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:       resolved_correct   No. Observations:                   35
Model:                          Logit   Df Residuals:                       33
Method:                           MLE   Df Model:                            1
Date:                Tue, 07 Jul 2026   Pseudo R-squ.:                0.004534
Time:                        12:19:52   Log-Likelihood:                -21.688
converged:                       True   LL-Null:                       -21.787
Covariance Type:            nonrobust   LLR p-value:                    0.6567
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.9163      0.483     -1.897      0.058      -1.863       0.030
Agent          0.3285      0.

4. Model comparisons

In [58]:
lr_test(RC1_A, RC3_A)

LR         0.778587
df         1.000000
p_value    0.377574
dtype: float64

In [59]:
lr_test(RC1_A, RC4_A)

LR         1.690503
df         2.000000
p_value    0.429449
dtype: float64

5. Summary Table

correct_summary_actors

In [60]:
goal_behaviour = ["Yes", "Yes", "Yes", "Yes", "Yes", "Yes", "Yes", "Yes"]

model_name = ["RC1_A", "RC2_A", "RC3_A", "RC3_A", "RC4_A", "RC4_A", "RC4_A", "RC5_A"]

outcome = ["resolved_correct","resolved_correct", "resolved_correct", "resolved_correct", "resolved_correct", "resolved_correct", "resolved_correct", "resolved_correct"]

subgroup = ["Actors", "Actors", "Actors", "Actors", "Actors", "Actors", "Actors", "Actors"]

formula = ["resolved_correct ~ Agent", "resolved_correct ~ Goal_Type", "resolved_correct ~ Agent + Goal_Type", "resolved_correct ~ Agent + Goal_Type", "resolved_correct ~ Agent * Goal_Type", "resolved_correct ~ Agent * Goal_Type", "resolved_correct ~ Agent * Goal_Type", "resolved_correct ~ Agent"]

predictor= ["Agent", "goal_non_frequent", "goal_non_frequent", "Agent",  "goal_non_frequent", "Agent", "Agent:goal_non_frequent", "Agent"]

coef = [-0.7438, -0.4590, -0.4618, -0.7456, -4.704e-16, -0.2513, -1.0116, 0.3285]

std_error = [0.523, 0.516, 0.525, 0.526, 0.713, 0.735, 1.065, 0.738]

coef_p = [0.155, 0.374, 0.379, 0.157, 1.000, 0.732, 0.342, 0.656]

LLR_p = [0.1512, 0.3723, 0.2419, 0.2419, 0.2897, 0.2897, 0.2897, 0.6567]

observations = [61, 61, 61, 61, 61, 61, 61, 35]

Takeaway = []

In [61]:
gb_correct_summary_actors = pd.DataFrame({
    "Goal_Behaviour": goal_behaviour,
    "Model": model_name,
    "Outcome": outcome,
    "Subgroup": subgroup,
    "Formula": formula, 
    "Predictor": predictor,
    "Coef": coef,
    "Std_err": std_error,
    "Coef_p": coef_p,
    "Model_LLR_p": LLR_p,
    "N": observations,
})

gb_correct_summary_actors

,Goal_Behaviour,Model,Outcome,Subgroup,Formula,Predictor,Coef,Std_err,Coef_p,Model_LLR_p,N
0,Yes,RC1_A,resolved_correct,Actors,resolved_correct ~ Agent,Agent,-7.438000e-01,0.523,0.155,0.1512,61
1,Yes,RC2_A,resolved_correct,Actors,resolved_correct ~ Goal_Type,goal_non_frequent,-4.590000e-01,0.516,0.374,0.3723,61
2,Yes,RC3_A,resolved_correct,Actors,resolved_correct ~ Agent + Goal_Type,goal_non_frequent,-4.618000e-01,0.525,0.379,0.2419,61
3,Yes,RC3_A,resolved_correct,Actors,resolved_correct ~ Agent + Goal_Type,Agent,-7.456000e-01,0.526,0.157,0.2419,61
4,Yes,RC4_A,resolved_correct,Actors,resolved_correct ~ Agent * Goal_Type,goal_non_frequent,-4.704000e-16,0.713,1.000,0.2897,61
5,Yes,RC4_A,resolved_correct,Actors,resolved_correct ~ Agent * Goal_Type,Agent,-2.513000e-01,0.735,0.732,0.2897,61
6,Yes,RC4_A,resolved_correct,Actors,resolved_correct ~ Agent * Goal_Type,Agent:goal_non_frequent,-1.011600e+00,1.065,0.342,0.2897,61
7,Yes,RC5_A,resolved_correct,Actors,resolved_correct ~ Agent,Agent,3.285000e-01,0.738,0.656,0.6567,35
